### 1. Cargar datos y seleccionar variables

In [0]:
df = spark.table("superstore_limpio")

df_modelo = df.select("Sales", "Quantity", "Discount", "Category",
                       "Sub_Category", "Region", "Segment", "Ship_Mode")

### 2. Codificar variables categóricas

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

categoricas = ["Category", "Sub_Category", "Region", "Segment", "Ship_Mode"]

indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in categoricas]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_ohe") for c in categoricas]

### 3. Ensamblar features

In [0]:
assembler = VectorAssembler(
    inputCols=["Quantity", "Discount"] + [c+"_ohe" for c in categoricas],
    outputCol="features"
)

### 4. Pipeline y split train/test

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.regression import GBTRegressor

pipeline = Pipeline(stages=indexers + encoders + [assembler])
df_prep = pipeline.fit(df_modelo).transform(df_modelo)

train, test = df_prep.randomSplit([0.8, 0.2], seed=42)

gbt = GBTRegressor(featuresCol="features", labelCol="Sales", maxIter=50)
modelo = gbt.fit(train)

### 5. Evaluar el modelo

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

predicciones = modelo.transform(test)

evaluator = RegressionEvaluator(labelCol="Sales", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predicciones)
r2 = evaluator.setMetricName("r2").evaluate(predicciones)
mae = evaluator.setMetricName("mae").evaluate(predicciones)

print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R2:   {r2:.3f}")

display(predicciones.select("Sales", "prediction"))

RMSE: 582.45
MAE:  193.18
R2:   0.187


Sales,prediction
0.836,-16.47078812284871
0.984,42.03720133431609
1.044,-16.47078812284871
1.167,-8.002273218175496
1.24,-3.130929866088916
1.344,-5.841331213655359
1.392,-18.260491878266222
1.476,45.15466496686613
1.68,2.4434954514132627
1.68,142.14133367764174


Databricks visualization. Run in Databricks to view.

### 6. Importancia de variables (para justificar resultados en el artículo)

In [0]:
import pandas as pd
importancias = pd.DataFrame({
    "feature_index": range(len(modelo.featureImportances)),
    "importancia": modelo.featureImportances.toArray()
}).sort_values("importancia", ascending=False)
display(importancias)

feature_index,importancia
0,0.38114072950106875
1,0.12369618128888149
21,0.07835787316318424
20,0.04141084024018224
25,0.037961642326195
24,0.03775299358358753
11,0.03185558274983978
30,0.03113911854634221
27,0.022889013331388135
15,0.0228602374843185


Databricks visualization. Run in Databricks to view.

### 7. Conclusión

El modelo de regresión (Gradient Boosted Trees) permite predecir el monto de ventas a partir de la 
cantidad comprada, el descuento aplicado y características del producto/pedido. Un RMSE de 582.45 
indica que, en promedio, las predicciones se desvían en esa magnitud del valor real de venta. Un R² 
de 0.187 muestra que el modelo explica solo el 18.7% de la variabilidad en las ventas, lo cual sugiere 
que estas variables por sí solas tienen poder predictivo limitado y que otros factores no incluidos en 
el modelo influyen más en el monto de venta.